### Notebook to make final adjustments to subglacial discharge CESM-WACCM processing

This notebook makes the following adjustments to the CESM-WACCM runoff files:
1. Making time axis consistent with TF files
2. Splitting into annual files
3. Fixing attributes (which look like they were stripped out at the bias correction stage)
4. Setting final file naming convention
5. Adding a 2300 file that is the same as 2299

### Setup

In [ ]:
# To run this notebook, uncomment one of the following setup blocks (historical, projection, or extension)

# historical
Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/historical/PROCESSEDmon/runoff/SDBN1'
Qfile = 'Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc'
Qroot = 'historical'
TFfile = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/historical/PROCESSEDmon/TF/TF_aQDM-ISMIP_Grid-CESM2-WACCM-1850_2014-PFromStep1-20251001.nc'
# projection
#Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/runoff/SDBN1'
#Qfile = 'Q_CESM2-WACCM_ssp585_SDBN1_2015_2100_biascorrected.nc'
#Qroot = 'ssp585'
#TFfile = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/TF/TF_aQDM-ISMIP_Grid-CESM2-WACCM-2015_2100-PFromStep1-20250926.nc'
# extension
# Qdir = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/runoff/SDBN1'
# Qfile = 'Q_CESM2-WACCM_ssp585_SDBN1_2101_2299_biascorrected.nc'
# Qroot = 'ssp585'
# TFfile = '/Users/dfelikso/Research/Data/ISMIP7/Forcing/CMIP/CESM2-WACCM/ssp585/PROCESSEDmon/TF/TF_aQDM-ISMIP_Grid-CESM2-WACCM-2101_2299-PFromStep1-20250926.nc'

# CESM-WACCM output file for year 2299
Q2299 = '~/Documents/CESM2-WACCM/ssp585/runoff/extension/Q_CESM2-WACCM_ssp585_SDBN1_2299.nc'


### Imports

In [ ]:
import xarray as xr
import cftime
import os
import numpy as np

In [ ]:
# code to get middle rather than end of month
days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

def to_exact_middle_of_month(time_array):
    new_times = []
    for t in time_array.values:
        year = t.year
        month = t.month
        # calculate middle day (round up)
        mid_day = int(np.ceil(days_in_month[month-1] / 2))
        new_time = cftime.DatetimeNoLeap(year, month, mid_day)
        new_times.append(new_time)
    return new_times

In [ ]:
# open file
ds_Q = xr.open_dataset(Qdir+Qfile,decode_times=False)

# get ocean thermal forcing for time axis
ds_TF = xr.open_dataset(TFfile)

# set subglacial discharge time axis to match thermal forcing time axis and move to middle of month
middle_times = to_exact_middle_of_month(ds_TF['time'])
ds_Q = ds_Q.assign_coords(time=middle_times)

# fix attributes that seem to have been stripped out when doing the bias correction
ds_Q['Q'].attrs['long_name'] = 'basin subglacial discharge, monthly mean'
ds_Q['Q'].attrs['units'] = 'm3 s-1'
ds_Q['Q'].attrs['grid_mapping'] = 'crs'
ds_Q['time'].attrs['standard_name'] = 'time'
ds_Q.attrs['Conventions'] = 'CF-1.8'

# write out files in annual chunks
startyear = ds_Q['time'].values[0].year
endyear = ds_Q['time'].values[-1].year
#endyear = 2018
for year in range(startyear,endyear+1):
    ds_Q_year = ds_Q.sel(time=slice(cftime.DatetimeNoLeap(year,1,1),cftime.DatetimeNoLeap(year,12,31)))
    outfile = Qdir+'Q_CESM2-WACCM_'+Qroot+'_SDBN1_'+str(year)+'.nc'
    # remove file if it exists
    if os.path.exists(outfile):
        os.remove(outfile)
    print('writing '+outfile)
    ds_Q_year.to_netcdf(outfile)

In [ ]:
# add a 2300 file that is the same as 2299 but with updated timestamp
ds_2299 = xr.open_dataset(Q2299)
ds_2300 = ds_2299.copy()

# replace year of time coordinate to 2300
new_times = []
for t in ds_2299['time'].values:
    new_time = cftime.DatetimeNoLeap(2300, t.month, t.day)
    new_times.append(new_time)
ds_2300 = ds_2300.assign_coords(time=new_times)

# save out
outfile = Q2299.replace('2299','2300')
if os.path.exists(outfile):
    os.remove(outfile)
ds_2300.to_netcdf(outfile)
